# GeoVision AI — Model Training Pipeline

This is the canonical, fully-corrected training pipeline used to produce both members of the final ensemble model:

1. **UNet / EfficientNet-B3** — set `CFG.MODEL_NAME = "Unet"`, `CFG.ENCODER_NAME = "efficientnet-b3"`
2. **UNet++ / EfficientNet-B5** — set `CFG.MODEL_NAME = "UnetPlusPlus"`, `CFG.ENCODER_NAME = "efficientnet-b5"`

Both were trained with this exact pipeline (same fixes, same data handling), only the two `CFG` lines above changed between runs. At inference/evaluation time, the two resulting checkpoints are softmax-averaged (see `notebooks/02_deployment_and_analysis.ipynb`).

**Pipeline design decisions, established through iterative debugging (documented for transparency — see the project report for the full experimental history):**
- `ignore_index` is passed to both the Cross-Entropy and Dice losses — omitting this silently trains on the "ignore" class and was an early bug.
- Loss is computed in FP32 even under AMP — Dice loss's epsilon underflows to 0 under FP16, which was producing intermittent `NaN` losses.
- Several shards are loaded together (`CFG.SHARD_GROUP_SIZE`) before building a batch, so each training step mixes patches from multiple geographic tiles rather than one — this resolved a significant epoch-to-epoch validation-IoU oscillation.
- An NDVI channel (computed from Red and NIR, after augmentation) is appended as a 5th input channel.
- A `WeightedRandomSampler` (computed per-shard from the loaded mask, not from a manifest join) oversamples rare classes (Built-up, Bare/sparse).
- Real checkpointing and early stopping, keyed on validation mean IoU.
- Evaluation supports test-time augmentation (flip-averaging) and multi-checkpoint ensembling.

**Requires:** the training dataset (Sentinel-2 RGB+NIR patches + ESA WorldCover 2021 labels, sharded as `.npz`) — see the main README for the dataset download link. Checkpoints produced by this notebook are not stored in this repository (too large for GitHub) — see `checkpoints/README.md` for the download link.

## Part 1 — Setup

In [ ]:
!pip install -q segmentation-models-pytorch albumentations

In [ ]:
import os
import gc
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import segmentation_models_pytorch as smp
import albumentations as A

In [ ]:
class CFG:

    # ------------------------
    # Reproducibility
    # ------------------------
    SEED = 42

    # ------------------------
    # Dataset
    # ------------------------
    # UPDATE THESE to your own dataset location before running.
    DATA_ROOT = Path("/kaggle/input/<your-dataset-slug>/prepared_dataset")
    SHARD_ROOT = DATA_ROOT / "shards"

    TRAIN_DIR = SHARD_ROOT / "train"
    VAL_DIR   = SHARD_ROOT / "val"
    TEST_DIR  = SHARD_ROOT / "test"

    # Used only to compute global class weights (not joined back to shards)
    MANIFEST_PATH = DATA_ROOT / "patch_manifest.csv"

    # ------------------------
    # Save
    # ------------------------
    SAVE_DIR = Path("/kaggle/working/checkpoints")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    # Name this run after whichever MODEL_NAME/ENCODER_NAME combination
    # you're training below -- this notebook is run twice (once per
    # ensemble member), and EXP_NAME keeps their checkpoints from
    # overwriting each other. Update it to match each run.
    EXP_NAME = "unetpp_efficientnet-b5"   # e.g. "unet_efficientnet-b3" for the other run

    # ------------------------
    # Dataset Parameters
    # ------------------------
    NUM_CLASSES = 7
    IGNORE_INDEX = 0
    IN_CHANNELS = 5   # 4 spectral bands (R, G, B, NIR) + 1 NDVI channel

    # ------------------------
    # Training
    # ------------------------
    BATCH_SIZE = 16
    NUM_WORKERS = 2

    NUM_EPOCHS = 15
    LR = 3e-4
    WEIGHT_DECAY = 1e-4

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP = True     # mixed precision -- loss is computed in FP32 regardless (see Part 3)

    PATIENCE = 8        # epochs without validation-mIoU improvement before early stopping

    # ------------------------
    # Model
    # ------------------------
    # Set these two lines to build either ensemble member:
    #   "Unet" / "efficientnet-b3"        -- first member
    #   "UnetPlusPlus" / "efficientnet-b5" -- second member
    MODEL_NAME = "UnetPlusPlus"
    ENCODER_NAME = "efficientnet-b5"
    ENCODER_WEIGHTS = "imagenet"

    # ------------------------
    # Scheduler
    # ------------------------
    SCHEDULER = "cosine"    # "cosine" or "onecycle"

    # ------------------------
    # Class imbalance handling
    # ------------------------
    # Class-weighted loss and the oversampling sampler both correct for the
    # same imbalance -- enabling both at full strength stacks the correction
    # and causes epoch-to-epoch validation instability. Use the sampler as
    # the primary lever and keep loss weighting off (or mild) by default.
    USE_CLASS_WEIGHTS = False
    CLASS_WEIGHT_CLIP = (0.5, 2.0)   # if enabled, clip inverse-sqrt weights to this range

    USE_WEIGHTED_SAMPLER = True
    RARE_CLASSES = (4, 5)   # Built-up, Bare/Sparse -- the underrepresented classes
    RARE_BOOST = 1.5

    GRAD_CLIP_NORM = 5.0

    # Several shards are loaded together before building each DataLoader, so
    # a batch mixes patches from multiple geographic tiles rather than one --
    # this is what resolved an earlier epoch-to-epoch validation-IoU
    # oscillation traced to poor batch composition.
    SHARD_GROUP_SIZE = 6

    # ------------------------
    # Evaluation (Part 5)
    # ------------------------
    USE_TTA_EVAL = True   # flip-averaged predictions at evaluation time only

    # Extra checkpoints to ensemble with this run's best checkpoint at eval
    # time (softmax-averaged). Leave empty to evaluate this run alone.
    ENSEMBLE_CHECKPOINTS = []

    # Evaluating on the full ~90k-patch train set is expensive and non-
    # standard (training metrics are normally tracked live, not re-run
    # after the fact). This samples a fixed subset of train shards instead,
    # for an honest, clearly-labeled reference number.
    TRAIN_EVAL_SHARD_SAMPLE = 10

print("Device:", CFG.DEVICE)

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG.SEED)

## Part 2 — Data pipeline

Key design choice: per-patch sampling weight is computed **directly from the loaded mask** at
train time, not read from `patch_manifest.csv`'s `sampling_weight` column. The manifest has no
shard/row index, so there's no reliable way to join a patch back to its position inside a
`.npz` shard. Computing the weight from the mask itself is exact and doesn't depend on that
join holding.

Augmentation is applied per-patch on the raw uint8 image, **before** the `/255` normalization
— this is what lets `RandomBrightnessContrast` and `GaussNoise` behave sensibly (they expect
realistic 0-255 pixel ranges, not pre-scaled floats).

In [ ]:
manifest = pd.read_csv(CFG.MANIFEST_PATH)
train_manifest = manifest[manifest["split"] == "train"]

class_cols = [
    "class_1_tree_cover_ratio", "class_2_low_vegetation_ratio", "class_3_cropland_ratio",
    "class_4_built_up_ratio", "class_5_bare_sparse_ratio", "class_6_water_wetland_ratio",
]
mean_ratios = train_manifest[class_cols].mean().values   # classes 1..6, in this order

inv_sqrt = 1.0 / np.sqrt(mean_ratios + 1e-6)
inv_sqrt = inv_sqrt / inv_sqrt.mean()                     # normalize weights around 1.0
inv_sqrt = np.clip(inv_sqrt, CFG.CLASS_WEIGHT_CLIP[0], CFG.CLASS_WEIGHT_CLIP[1])

class_weights = torch.zeros(CFG.NUM_CLASSES, dtype=torch.float32)
class_weights[1:] = torch.tensor(inv_sqrt, dtype=torch.float32)
class_weights[CFG.IGNORE_INDEX] = 0.0
class_weights = class_weights.to(CFG.DEVICE)

print("Class weights (index 0 = ignore, 1..6 = semantic classes):")
print(class_weights.cpu().numpy())

In [ ]:
def load_shard_group(shard_paths):
    """Loads and concatenates several shards into one larger in-RAM pool,
    so a DataLoader built on top of it mixes patches across many tiles
    instead of just the ~1,960 patches in a single shard."""
    imgs_list, masks_list = [], []
    for p in shard_paths:
        imgs, masks = load_full_shard_raw(p)
        imgs_list.append(imgs)
        masks_list.append(masks)
    return np.concatenate(imgs_list, axis=0), np.concatenate(masks_list, axis=0)


train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
    A.GaussNoise(var_limit=(5.0, 20.0), p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15,
                        border_mode=0, p=0.4),
])


def add_ndvi_channel(img_uint8):
    """Appends an NDVI band computed from Red (ch 0) and NIR (ch 3), rescaled
    to uint8 [0, 255] so it stacks cleanly with the other channels."""
    img = img_uint8.astype(np.float32)
    red, nir = img[..., 0], img[..., 3]
    ndvi = (nir - red) / (nir + red + 1e-6)                        # range [-1, 1]
    ndvi_scaled = ((ndvi + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    return np.dstack([img_uint8, ndvi_scaled])


class ShardDataset(Dataset):
    """Wraps one loaded shard (uint8 images, int64 masks). Albumentations is
    applied per-patch BEFORE normalization/transpose; NDVI is derived AFTER
    augmentation so it always matches the final RGB+NIR bands."""

    def __init__(self, images_uint8, masks_int64, transform=None):
        self.images = images_uint8   # (N, H, W, 4) uint8
        self.masks = masks_int64     # (N, H, W) int64
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        mask = self.masks[idx]

        if self.transform is not None:
            augmented = self.transform(image=img, mask=mask)
            img, mask = augmented["image"], augmented["mask"]

        if CFG.IN_CHANNELS == 5:
            img = add_ndvi_channel(img)

        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))   # HWC -> CHW
        mask = mask.astype(np.int64)

        return torch.from_numpy(img), torch.from_numpy(mask)


def load_full_shard_raw(shard_path):
    """Loads a shard WITHOUT normalizing -- normalization now happens per-item
    in ShardDataset, after augmentation."""
    with np.load(shard_path, allow_pickle=True) as data:
        images = data["images"]              # (N, H, W, C) uint8
        masks = data["masks"].astype(np.int64)
    return images, masks


def compute_patch_weights(masks, rare_classes=CFG.RARE_CLASSES, rare_boost=CFG.RARE_BOOST, base=1.0):
    """Per-patch sampling weight computed directly from the mask -- boosts
    patches containing Built-up / Bare-sparse pixels."""
    total_pixels = masks.shape[1] * masks.shape[2]
    rare_mask = np.isin(masks, rare_classes)
    rare_frac = rare_mask.reshape(masks.shape[0], -1).mean(axis=1)
    return (base + rare_boost * rare_frac).astype(np.float32)


def make_shard_loader(images, masks, batch_size, shuffle, transform=None, use_weighted_sampler=False):
    ds = ShardDataset(images, masks, transform=transform)

    if shuffle and use_weighted_sampler:
        weights = compute_patch_weights(masks)
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
        loader = DataLoader(
            ds, batch_size=batch_size, sampler=sampler,
            num_workers=CFG.NUM_WORKERS, pin_memory=(CFG.DEVICE == "cuda"),
            drop_last=False,
        )
    else:
        loader = DataLoader(
            ds, batch_size=batch_size, shuffle=shuffle,
            num_workers=CFG.NUM_WORKERS, pin_memory=(CFG.DEVICE == "cuda"),
            drop_last=False,
        )
    return loader

In [ ]:
train_shards = sorted(CFG.TRAIN_DIR.glob("*.npz"))
val_shards   = sorted(CFG.VAL_DIR.glob("*.npz"))
test_shards  = sorted(CFG.TEST_DIR.glob("*.npz"))

print("Train shards:", len(train_shards))
print("Val shards:", len(val_shards))
print("Test shards:", len(test_shards))

In [ ]:
# Sanity check: load one shard, confirm shapes/classes, and that the
# rare-class boosting produces sensible weights.
sample_images, sample_masks = load_full_shard_raw(train_shards[0])

print("images:", sample_images.shape, sample_images.dtype)
print("masks:", sample_masks.shape, sample_masks.dtype)
print("mask unique:", np.unique(sample_masks))
print("mean patch weight (rare-class boost):", compute_patch_weights(sample_masks).mean())

## Part 3 — Model, loss, optimizer, scheduler

In [ ]:
ModelClass = getattr(smp, CFG.MODEL_NAME)

model = ModelClass(
    encoder_name=CFG.ENCODER_NAME,
    encoder_weights=CFG.ENCODER_WEIGHTS,
    in_channels=CFG.IN_CHANNELS,
    classes=CFG.NUM_CLASSES,
    activation=None,
).to(CFG.DEVICE)

print(model.__class__.__name__, "| Encoder:", CFG.ENCODER_NAME)

In [ ]:
ce_loss = nn.CrossEntropyLoss(
    weight=class_weights if CFG.USE_CLASS_WEIGHTS else None,
    ignore_index=CFG.IGNORE_INDEX,
)

dice_loss = smp.losses.DiceLoss(
    mode="multiclass",
    from_logits=True,
    ignore_index=CFG.IGNORE_INDEX,
)

def criterion(preds, masks):
    # Guards against the rare case where an entire batch has no valid
    # (non-ignore) pixels -- CrossEntropyLoss means over 0 elements = nan.
    valid_pixels = (masks != CFG.IGNORE_INDEX).sum()
    if valid_pixels == 0:
        return torch.zeros((), device=preds.device, dtype=preds.dtype, requires_grad=True)
    return ce_loss(preds, masks) + dice_loss(preds, masks)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.LR,
    weight_decay=CFG.WEIGHT_DECAY,
)

def build_scheduler(optimizer, steps_per_epoch=None):
    if CFG.SCHEDULER == "cosine":
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CFG.NUM_EPOCHS, eta_min=1e-6
        )
        return sch, "epoch"        # .step() once per epoch
    elif CFG.SCHEDULER == "onecycle":
        sch = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=CFG.LR, epochs=CFG.NUM_EPOCHS,
            steps_per_epoch=steps_per_epoch, pct_start=0.3,
        )
        return sch, "batch"        # .step() every batch
    else:
        raise ValueError(f"Unknown scheduler: {CFG.SCHEDULER}")

# Only needed for OneCycleLR -- reuses train_manifest, no extra shard I/O.
steps_per_epoch = (len(train_manifest) // CFG.BATCH_SIZE) + 1 if CFG.SCHEDULER == "onecycle" else None

scheduler, scheduler_mode = build_scheduler(optimizer, steps_per_epoch)

scaler = torch.amp.GradScaler("cuda", enabled=(CFG.DEVICE == "cuda" and CFG.USE_AMP))

In [ ]:
def compute_batch_iou(preds, masks, num_classes=7, ignore_index=0):
    preds = preds.reshape(-1)
    masks = masks.reshape(-1)

    valid = masks != ignore_index
    preds = preds[valid]
    masks = masks[valid]

    ious = []

    for cls in range(num_classes):
        if cls == ignore_index:
            continue

        pred_cls = preds == cls
        mask_cls = masks == cls

        intersection = (pred_cls & mask_cls).sum().item()
        union = (pred_cls | mask_cls).sum().item()

        if union == 0:
            ious.append(np.nan)
        else:
            ious.append(intersection / union)

    return ious

def compute_confusion_counts(preds, masks, num_classes, ignore_index):
    """Accumulatable per-class TP/FP/FN/support counts (pixel-level),
    excluding the ignore class -- used for precision/recall/F1/FWIoU."""
    preds = preds.reshape(-1)
    masks = masks.reshape(-1)
    valid = masks != ignore_index
    preds = preds[valid]
    masks = masks[valid]

    tp = np.zeros(num_classes)
    fp = np.zeros(num_classes)
    fn = np.zeros(num_classes)
    support = np.zeros(num_classes)

    for cls in range(num_classes):
        if cls == ignore_index:
            continue
        pred_cls = preds == cls
        mask_cls = masks == cls
        tp[cls] = (pred_cls & mask_cls).sum()
        fp[cls] = (pred_cls & ~mask_cls).sum()
        fn[cls] = (~pred_cls & mask_cls).sum()
        support[cls] = mask_cls.sum()

    return tp, fp, fn, support


def compute_metrics_from_counts(tp, fp, fn, support, num_classes, ignore_index):
    """Turns accumulated confusion counts into per-class IoU/Precision/Recall/F1
    plus corpus-level means and Frequency-Weighted IoU (FWIoU)."""
    iou = np.full(num_classes, np.nan)
    precision = np.full(num_classes, np.nan)
    recall = np.full(num_classes, np.nan)
    f1 = np.full(num_classes, np.nan)

    for cls in range(num_classes):
        if cls == ignore_index:
            continue
        denom_iou = tp[cls] + fp[cls] + fn[cls]
        iou[cls] = tp[cls] / denom_iou if denom_iou > 0 else np.nan

        denom_p = tp[cls] + fp[cls]
        precision[cls] = tp[cls] / denom_p if denom_p > 0 else np.nan

        denom_r = tp[cls] + fn[cls]
        recall[cls] = tp[cls] / denom_r if denom_r > 0 else np.nan

        if not np.isnan(precision[cls]) and not np.isnan(recall[cls]) and (precision[cls] + recall[cls]) > 0:
            f1[cls] = 2 * precision[cls] * recall[cls] / (precision[cls] + recall[cls])

    valid_mask = np.array([c != ignore_index for c in range(num_classes)])
    mean_iou = np.nanmean(iou[valid_mask])
    mean_precision = np.nanmean(precision[valid_mask])
    mean_recall = np.nanmean(recall[valid_mask])
    mean_f1 = np.nanmean(f1[valid_mask])

    class_freq = support[valid_mask] / support[valid_mask].sum()
    valid_iou = iou[valid_mask]
    has_iou = ~np.isnan(valid_iou)
    fwiou = np.sum(valid_iou[has_iou] * class_freq[has_iou]) / np.sum(class_freq[has_iou])

    return {
        "iou": iou, "precision": precision, "recall": recall, "f1": f1,
        "mean_iou": mean_iou, "mean_precision": mean_precision,
        "mean_recall": mean_recall, "mean_f1": mean_f1, "fwiou": fwiou,
    }


def predict_tta(model, imgs):
    """Flip-averaged test-time augmentation -- eval only, no training-time cost."""
    preds = model(imgs)
    preds = preds + torch.flip(model(torch.flip(imgs, dims=[3])), dims=[3])
    preds = preds + torch.flip(model(torch.flip(imgs, dims=[2])), dims=[2])
    return preds / 3.0


def predict_ensemble(models, imgs, use_tta=False):
    """Softmax-averages predictions across multiple loaded models (checkpoints)."""
    probs_sum = None
    for m in models:
        logits = predict_tta(m, imgs) if use_tta else m(imgs)
        probs = torch.softmax(logits, dim=1)
        probs_sum = probs if probs_sum is None else probs_sum + probs
    return probs_sum / len(models)


def evaluate_full(models, shard_paths, device, num_classes=CFG.NUM_CLASSES,
                   ignore_index=CFG.IGNORE_INDEX, use_tta=False, desc="Evaluating"):
    """Full precision/recall/F1/IoU/FWIoU evaluation across a set of shards.
    `models` is a list -- pass a single-element list for one checkpoint, or
    several loaded models to ensemble."""
    for m in models:
        m.eval()

    tp_total = np.zeros(num_classes)
    fp_total = np.zeros(num_classes)
    fn_total = np.zeros(num_classes)
    support_total = np.zeros(num_classes)

    with torch.no_grad():
        for shard_path in tqdm(shard_paths, desc=desc):
            images, masks = load_full_shard_raw(shard_path)
            loader = make_shard_loader(images, masks, CFG.BATCH_SIZE, shuffle=False, transform=None)

            for imgs, target_masks in loader:
                imgs = imgs.to(device, non_blocking=True)
                target_masks_np = target_masks.numpy()

                with torch.amp.autocast("cuda", enabled=(device == "cuda" and CFG.USE_AMP)):
                    if len(models) > 1:
                        probs = predict_ensemble(models, imgs, use_tta=use_tta)
                        pred_labels = torch.argmax(probs, dim=1).cpu().numpy()
                    else:
                        preds = predict_tta(models[0], imgs) if use_tta else models[0](imgs)
                        pred_labels = torch.argmax(preds, dim=1).cpu().numpy()

                tp, fp, fn, support = compute_confusion_counts(pred_labels, target_masks_np, num_classes, ignore_index)
                tp_total += tp
                fp_total += fp
                fn_total += fn
                support_total += support

            del images, masks, loader
            gc.collect()

    return compute_metrics_from_counts(tp_total, fp_total, fn_total, support_total, num_classes, ignore_index)


def print_metrics(name, metrics, class_names):
    print(f"\n=== {name} ===")
    print(f"Mean IoU:  {metrics['mean_iou']:.4f}")
    print(f"FWIoU:     {metrics['fwiou']:.4f}")
    print(f"Precision: {metrics['mean_precision']:.4f}")
    print(f"Recall:    {metrics['mean_recall']:.4f}")
    print(f"F1:        {metrics['mean_f1']:.4f}")
    for i, cname in enumerate(class_names, start=1):
        print(f"  {cname:16s} IoU={metrics['iou'][i]:.4f}  P={metrics['precision'][i]:.4f}  "
              f"R={metrics['recall'][i]:.4f}  F1={metrics['f1'][i]:.4f}")

In [ ]:
class_names = [
    "Tree cover",
    "Low vegetation",
    "Cropland",
    "Built-up",
    "Bare / sparse",
    "Water / wetland",
]

## Part 4 — Training loop 

The loop:
- writes `best_model.pth` when validation mIoU improves,
- uses `PATIENCE`/`no_improve` to stop early,
- steps the scheduler at the right granularity depending on whether it's Cosine (per-epoch) or OneCycle (per-batch).

In [ ]:
def train_one_epoch(model, shard_paths, optimizer, scheduler, scheduler_mode, scaler, device):
    model.train()
    running_loss = 0.0
    total_batches = 0

    shard_paths = list(shard_paths)
    random.shuffle(shard_paths)

    groups = [shard_paths[i:i + CFG.SHARD_GROUP_SIZE]
              for i in range(0, len(shard_paths), CFG.SHARD_GROUP_SIZE)]

    group_pbar = tqdm(groups, desc="Training shard groups")

    for group in group_pbar:
        images, masks = load_shard_group(group)
        loader = make_shard_loader(
            images, masks, CFG.BATCH_SIZE, shuffle=True,
            transform=train_transform,
            use_weighted_sampler=CFG.USE_WEIGHTED_SAMPLER,
        )

        for imgs, target_masks in loader:
            imgs = imgs.to(device, non_blocking=True)
            target_masks = target_masks.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=(device == "cuda" and CFG.USE_AMP)):
                preds = model(imgs)

            # loss computed in FP32 -- Dice's eps underflows to 0 in FP16,
            # which produced the "Val Loss: nan" you saw
            loss = criterion(preds.float(), target_masks)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG.GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()

            if scheduler_mode == "batch":
                scheduler.step()

            running_loss += loss.item()
            total_batches += 1

        del images, masks, loader
        gc.collect()

    return running_loss / max(total_batches, 1)

In [ ]:
def validate_one_epoch(model, shard_paths, device, num_classes=7, ignore_index=0):
    model.eval()
    running_loss = 0.0
    total_batches = 0
    all_ious = []

    shard_pbar = tqdm(shard_paths, desc="Validation shards")

    with torch.no_grad():
        for shard_path in shard_pbar:
            images, masks = load_full_shard_raw(shard_path)
            loader = make_shard_loader(images, masks, CFG.BATCH_SIZE, shuffle=False, transform=None)

            for imgs, target_masks in loader:
                imgs = imgs.to(device, non_blocking=True)
                target_masks = target_masks.to(device, non_blocking=True)

                with torch.amp.autocast("cuda", enabled=(device == "cuda" and CFG.USE_AMP)):
                    preds = model(imgs)

                loss = criterion(preds.float(), target_masks)

                running_loss += loss.item()
                total_batches += 1

                pred_labels = torch.argmax(preds, dim=1)
                batch_ious = compute_batch_iou(
                    pred_labels.cpu(), target_masks.cpu(),
                    num_classes=num_classes, ignore_index=ignore_index,
                )
                all_ious.append(batch_ious)

            del images, masks, loader
            gc.collect()

    all_ious = np.array(all_ious, dtype=np.float32)
    class_iou = np.nanmean(all_ious, axis=0)
    mean_iou = np.nanmean(class_iou)

    return running_loss / max(total_batches, 1), class_iou, mean_iou

In [ ]:
best_val_iou = -1.0
history = []
no_improve = 0

for epoch in range(CFG.NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{CFG.NUM_EPOCHS}")
    start_time = time.time()

    train_loss = train_one_epoch(
        model=model, shard_paths=train_shards, optimizer=optimizer,
        scheduler=scheduler, scheduler_mode=scheduler_mode,
        scaler=scaler, device=CFG.DEVICE,
    )

    val_loss, class_iou, mean_iou = validate_one_epoch(
        model=model, shard_paths=val_shards, device=CFG.DEVICE,
        num_classes=CFG.NUM_CLASSES, ignore_index=CFG.IGNORE_INDEX,
    )

    if scheduler_mode == "epoch":
        scheduler.step()

    elapsed = time.time() - start_time

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Mean IoU:   {mean_iou:.4f}")
    print(f"Epoch Time: {elapsed/60:.2f} min")
    print(f"Learning Rate: {optimizer.param_groups[0]["lr"]:.2e}")
    for i, score in enumerate(class_iou):
        print(f"  {class_names[i]} IoU: {score:.4f}")

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "mean_iou": float(mean_iou),
        "lr": optimizer.param_groups[0]["lr"],
        **{f"class_{i}_iou": float(class_iou[i]) for i in range(len(class_iou))},
    })

    # --- checkpointing + early stopping ---
    if mean_iou > best_val_iou:
        best_val_iou = mean_iou
        no_improve = 0
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "mean_iou": mean_iou,
            "class_iou": class_iou.tolist(),
        }, CFG.SAVE_DIR / "best_model.pth")
        print(f"  -> New best model saved (mIoU={mean_iou:.4f})")
    else:
        no_improve += 1
        print(f"  -> No improvement ({no_improve}/{CFG.PATIENCE})")
        if no_improve >= CFG.PATIENCE:
            print("Early stopping triggered.")
            break

In [ ]:
LAST_MODEL_PATH = CFG.SAVE_DIR / f"{CFG.EXP_NAME}_last.pth"
torch.save(model.state_dict(), LAST_MODEL_PATH)
print("Saved last-epoch weights to", LAST_MODEL_PATH)

In [ ]:
history_df = pd.DataFrame(history)
history_df.to_csv(CFG.SAVE_DIR / f"{CFG.EXP_NAME}_training_history.csv", index=False)
history_df

## Part 5 — Final evaluation on the test set

Loads the best checkpoint (by validation mIoU) and reports per-class + mean IoU on the held-out
test shards. Remember the caveat from Part 1: the test tiles are Built-up-poor relative to
train/val, so treat the Built-up test IoU specifically as a lower-confidence number.

In [ ]:
# weights_only=False: safe here since this is your own checkpoint from this
# same session/notebook, not an untrusted download.
best_ckpt = torch.load(CFG.SAVE_DIR / "best_model.pth", map_location=CFG.DEVICE, weights_only=False)
model.load_state_dict(best_ckpt["model_state_dict"])
print("Loaded best model from epoch:", best_ckpt["epoch"], "| val mIoU:", best_ckpt["mean_iou"])

models_for_eval = [model]

# Optional ensembling: load any extra checkpoints listed in CFG.ENSEMBLE_CHECKPOINTS
# and average their softmax outputs with the primary model above.
for ckpt_path in CFG.ENSEMBLE_CHECKPOINTS:
    extra_model = ModelClass(
        encoder_name=CFG.ENCODER_NAME, encoder_weights=None,
        in_channels=CFG.IN_CHANNELS, classes=CFG.NUM_CLASSES, activation=None,
    ).to(CFG.DEVICE)
    ckpt = torch.load(ckpt_path, map_location=CFG.DEVICE, weights_only=False)
    extra_model.load_state_dict(ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt)
    extra_model.eval()
    models_for_eval.append(extra_model)
    print("Added to ensemble:", ckpt_path)

# --- Test set: full evaluation (this is your reportable test number) ---
test_metrics = evaluate_full(
    models_for_eval, test_shards, CFG.DEVICE,
    use_tta=CFG.USE_TTA_EVAL, desc="Evaluating test set",
)
print_metrics("Test", test_metrics, class_names)

# --- Train set: evaluated on a fixed sample of shards, not the full ~90k patches ---
# (full-train re-evaluation is expensive and non-standard; this gives an honest,
# clearly-labeled reference point rather than an inflated full-corpus number).
train_eval_shards = train_shards[:CFG.TRAIN_EVAL_SHARD_SAMPLE]
train_metrics = evaluate_full(
    models_for_eval, train_eval_shards, CFG.DEVICE,
    use_tta=CFG.USE_TTA_EVAL, desc=f"Evaluating train sample ({len(train_eval_shards)} shards)",
)
print_metrics(f"Train (sample of {len(train_eval_shards)} shards)", train_metrics, class_names)
